# 01 — Data Collection


## 1. Install Libraries



In [1]:
# !pip install datasets requests tqdm -q


## 2. Imports and Folder Setup



In [ ]:
import os
import json
import requests
from datasets import load_dataset
from tqdm import tqdm

# Create folder structure
# This is where all raw data will live
os.makedirs('data/raw/pdfs', exist_ok=True)

print('Folders created:')



C:\Users\yassin\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.1.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(
C:\Users\yassin\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Folders created:
  data/raw/         <- tweets will go here
  data/raw/pdfs/    <- PDFs will go here


## 3. Load Tweets Dataset


In [2]:
# load_dataset takes the repository name from HuggingFace
# Format: 'username/dataset-name'
print('Loading PoliticalTweets dataset...')
print('This downloads ~50MB, takes 1-2 minutes on first run')
print('On subsequent runs it loads from cache instantly')

ds = load_dataset('Jacobvs/PoliticalTweets')

print()
print('Dataset loaded.')
print('Type:', type(ds))
print('Splits available:', list(ds.keys()))


Loading PoliticalTweets dataset...
This downloads ~50MB, takes 1-2 minutes on first run
On subsequent runs it loads from cache instantly


Generating train split: 100%|██████████| 190491/190491 [00:00<00:00, 259586.34 examples/s]



Dataset loaded.
Type: <class 'datasets.dataset_dict.DatasetDict'>
Splits available: ['train']


## 4. Explore the Dataset



In [ ]:

train = ds['train']

print('=== Dataset Info ===')
print(f'Total rows: {len(train):,}')
print(f'Columns: {train.column_names}')
print()

import pandas as pd
df = train.to_pandas()

print('=== First 3 rows ===')
print(df[['text', 'party', 'labels']].head(3).to_string())
print()

print('=== Label distribution ===')
print(df['party'].value_counts())
print()

df['word_count'] = df['text'].str.split().str.len()
print('=== Text length (words) ===')
print(df['word_count'].describe())


=== Dataset Info ===
Total rows: 190,491
Columns: ['index', 'date', 'id', 'username', 'text', 'party', 'labels']

=== First 3 rows ===
                                                                                                                                                                                                                                                                                     text       party  labels
0    Happy th birthday to the @USNavy! The strength, dedication, and skill of our Sailors including those at Portsmouth Naval Shipyard help keep this country safe, secure, and free. Today we recognize and celebrate their incredible service. #246NavyBirthday https://t.co/GuHEDMApke    Democrat       1
1  The greatest generation's investment in infrastructure made us the envy of the world. But now we've gone almost an entire lifetime without making any significant investments in the NEXT generation of American infrastructure. It's time for that to change. htt

In [ ]:


print('=== 5 Democrat tweets ===')
dem_samples = df[df['party'] == 'Democrat']['text'].head(5).tolist()
for i, t in enumerate(dem_samples):
    print(f'{i+1}. {t[:150]}')
    print()

print('=== 5 Republican tweets ===')
rep_samples = df[df['party'] == 'Republican']['text'].head(5).tolist()
for i, t in enumerate(rep_samples):
    print(f'{i+1}. {t[:150]}')
    print()


=== 5 Democrat tweets ===
1. Happy th birthday to the @USNavy! The strength, dedication, and skill of our Sailors including those at Portsmouth Naval Shipyard help keep this count

2. The greatest generation's investment in infrastructure made us the envy of the world. But now we've gone almost an entire lifetime without making any 

3. / To get lasting change we cant just lock up those convicted of these crimes, but must also work to combat bias and  bigotry. The NO HATE Act would al

4. The #ForthePeopleAct includes reforms that are popular with Americans no matter what party theyre from: Expanded early voting Automatic voter registra

5. Todays strong, bipartisan vote is just the beginningI'm going to continue fighting for more investments necessary to meet our nations challenges.

=== 5 Republican tweets ===
1. Thanks to @SenTedCruz and  @SenatorWarnock, the Infrastructure Investment and Jobs Act authorizes the Interstate East-West corridor across TX, LA, MS,

2. Today were celebra

In [ ]:
short = df[df['word_count'] <= 15]
print(f'Tweets with <= 15 words: {len(short):,} ({len(short)/len(df)*100:.1f}%)')
print()
print('Examples of short tweets (likely neutral):')
for t in short['text'].head(5).tolist():
    print(f'  -> {t}')


Tweets with <= 15 words: 21,387 (11.2%)

Examples of short tweets (likely neutral):
  -> Today were celebrating years of the Hoosier state. Happy birthday Indiana! https://t.co/gjzh3dHnIb
  -> Today we start anew.
  -> Because its right https://t.co/4PXNIHVSWM
  -> Three words to describe what we've seen from President Biden's administration: Boring but radical.
  -> This is abhorrent. Our country should be better than this. https://t.co/xrLTDW8vOB


## 5. Save Tweets as JSONL



In [ ]:
output_path = 'data/raw/tweets_raw.jsonl'

df_save = df[['text', 'party']].copy()
df_save = df_save.rename(columns={'party': 'ideology'})
df_save['ideology'] = df_save['ideology'].str.lower()

# Save as JSONL
with open(output_path, 'w', encoding='utf-8') as f:
    for _, row in tqdm(df_save.iterrows(), total=len(df_save), desc='Saving'):
        f.write(json.dumps({
            'text': row['text'],
            'ideology': row['ideology']
        }, ensure_ascii=False) + '\n')

print(f'Saved {len(df_save):,} rows to {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB')


Saving: 100%|██████████| 190491/190491 [00:04<00:00, 38982.29it/s]

Saved 190,491 rows to data/raw/tweets_raw.jsonl
File size: 39.1 MB


In [ ]:
print('First 3 lines of saved file:')
with open(output_path) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        row = json.loads(line)
        print(f'ideology: {row["ideology"]}')
        print(f'text:     {row["text"][:100]}')
        print()


First 3 lines of saved file:
ideology: democrat
text:     Happy th birthday to the @USNavy! The strength, dedication, and skill of our Sailors including those

ideology: democrat
text:     The greatest generation's investment in infrastructure made us the envy of the world. But now we've 

ideology: republican
text:     Thanks to @SenTedCruz and  @SenatorWarnock, the Infrastructure Investment and Jobs Act authorizes th



## 6. Download Party Platform PDFs

**What are these PDFs?**

Official party documents published every 4 years.
They contain the party's position on every major topic.



In [ ]:
PDFS = {
    'democratic_2024.pdf':  'https://democrats.org/wp-content/uploads/2025/07/2024-Democratic-Party-Platform.pdf',
    'republican_2024.pdf':  'https://prod-static.gop.com/media/RNC2024-Platform.pdf',
}

def download_pdf(url, save_path):

    #Download a PDF from a URL and save it to disk.

    print(f'Downloading {os.path.basename(save_path)}...')
    
    try:
        response = requests.get(
            url,
            stream=True,
            timeout=30,
            # pretend to be a browser
            headers={'User-Agent': 'Mozilla/5.0'}
        )
        response.raise_for_status()
        
        #chunks of 8KB
        with open(save_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        size_mb = os.path.getsize(save_path) / 1024 / 1024
        print(f'  Saved: {save_path} ({size_mb:.1f} MB)')
        return True
    
    except Exception as e:
        print(f'  FAILED: {e}')
        return False


# Download all PDFs
results = {}
for filename, url in PDFS.items():
    path = f'data/raw/pdfs/{filename}'
    
    # Skip if already downloaded — don't re-download
    if os.path.exists(path):
        print(f'Already exists, skipping: {filename}')
        results[filename] = True
        continue
    
    results[filename] = download_pdf(url, path)

print()
print('=== Download Summary ===')
for fname, success in results.items():
    status = 'OK' if success else 'FAILED'
    print(f'  [{status}] {fname}')


  Saved: data/raw/pdfs/democratic_2024.pdf (0.9 MB)
  Saved: data/raw/pdfs/republican_2024.pdf (1.3 MB)

=== Download Summary ===
  [OK] democratic_2024.pdf
  [OK] republican_2024.pdf


In [ ]:
print('Verifying PDFs...')
for filename in PDFS.keys():
    path = f'data/raw/pdfs/{filename}'
    if not os.path.exists(path):
        print(f'  MISSING: {filename}')
        continue
    
    # Read first 4 bytes
    with open(path, 'rb') as f:
        header = f.read(4)
    
    size_mb = os.path.getsize(path) / 1024 / 1024
    is_pdf = header == b'%PDF'
    status = 'VALID PDF' if is_pdf else 'CORRUPTED'
    print(f'  [{status}] {filename} ({size_mb:.1f} MB)')


Verifying PDFs...
  [VALID PDF] democratic_2024.pdf (0.9 MB)
  [VALID PDF] republican_2024.pdf (1.3 MB)
